# 04 · El modelo no es neutro

## Propósito

Este es el cuaderno bisagra de la unidad.

Hasta acá usamos `es_core_news_lg` sin preguntarnos por qué. Ahora vamos a
comparar los tres modelos españoles de spaCy, ver qué cambia entre uno y otro,
y —más importante— **qué no se arregla con ninguno**.

Después vamos a mirar de verdad la salida del NER sobre nuestro corpus y medir
cuánto se puede confiar en ella.

## Qué consume

- `datos/corpus.jsonl`
- Los tres modelos: `es_core_news_sm`, `es_core_news_md`, `es_core_news_lg`

Si te falta alguno:

```
python -m spacy download es_core_news_sm
python -m spacy download es_core_news_md
```

## Celda de reanudación

In [ ]:
# spacy: la librería de análisis lingüístico.
import spacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# Counter: para contar.
from collections import Counter

# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

# Cargamos los tres modelos. Tarda un rato: son 700 MB entre los tres.
modelos = {}
modelos["sm"] = spacy.load("es_core_news_sm")
modelos["md"] = spacy.load("es_core_news_md")
modelos["lg"] = spacy.load("es_core_news_lg")

claves_de_modelos = modelos.keys()  # keys(): las claves del diccionario
nombres_de_modelos = list(claves_de_modelos)  # list(): las convierte en una lista
print("modelos cargados:", nombres_de_modelos)

# vocab.vectors: la tabla de vectores del modelo. shape[0]: cuántas filas (palabras) tiene
tabla_sm = modelos["sm"].vocab.vectors
tabla_lg = modelos["lg"].vocab.vectors
vectores_sm = tabla_sm.shape[0]
vectores_lg = tabla_lg.shape[0]

if vectores_sm >= vectores_lg:
    print("ATENCIÓN: el modelo sm debería tener menos vectores que el lg. Si no, alguno de los")
    print("tres no se cargó y estarías comparando un modelo consigo mismo.")

## 1. En qué se diferencian

El nombre dice el tamaño, pero no dice qué se gana. Preguntémosle a cada modelo.

Pongamos los tres modelos lado a lado: cuántos vectores traen y cuántos componentes
tiene su pipeline. Lo que pesan en disco ya lo viste en el `02`: 16 MB el `sm`,
54 MB el `md` y 631 MB el `lg`.

In [ ]:
filas = []
for etiqueta in ["sm", "md", "lg"]:
    nlp = modelos[etiqueta]

    cantidad_de_vectores = nlp.vocab.vectors.shape[0]
    cantidad_de_componentes = len(nlp.pipe_names)

    fila = {
        "modelo": etiqueta,
        "vectores": cantidad_de_vectores,
        "componentes": cantidad_de_componentes,
    }
    filas.append(fila)

pd.DataFrame(filas)

Los tres tienen los mismos seis componentes. Lo que cambia es el tamaño de
los pesos y, sobre todo, **los vectores de palabras**: `sm` no tiene ninguno,
`md` tiene 20.000 y `lg` tiene 500.000. Volvemos sobre eso en el cuaderno `07`.

¿Y la velocidad? La intuición dice que el grande es más lento, pero sobre los
mismos párrafos los tres tardan prácticamente lo mismo. **Elegir el modelo chico
no te compra velocidad; te cuesta calidad.** El único costo real del grande es el
disco y la memoria.

## 2. El voseo, con los tres modelos

Ahora la pregunta que importa para nuestro corpus. Los modelos españoles de
spaCy se entrenaron principalmente con AnCora y WikiNER: corpus de español
peninsular, prosa periodística formal.

Nuestro corpus es un blog rioplatense. Veamos qué pasa.

In [ ]:
frase = "Vos tenés razón y sabés lo que hacés. Vos sos mi colega, che."

analisis = {}
for etiqueta in ["sm", "md", "lg"]:
    modelo = modelos[etiqueta]
    analisis[etiqueta] = modelo(frase)

filas = []
for token in analisis["sm"]:

    # Descartamos los signos de puntuación.
    if not token.is_punct:
        fila = {"token": token.text}
        for etiqueta in ["sm", "md", "lg"]:
            # token.i es la posición de ese token dentro del Doc.
            otro = analisis[etiqueta][token.i]
            fila[etiqueta] = otro.pos_ + " / " + otro.lemma_
        filas.append(fila)

pd.DataFrame(filas)

Leelo con calma, porque acá está el contenido del cuaderno.

**Lo que se arregla al subir de modelo:** con `sm`, `tenés` es un adjetivo,
`sabés` un sustantivo y `sos` un nombre propio. Con `md` y `lg` los tres son
verbos. `che` pasa de sustantivo a interjección. `Vos` pasa de nombre propio a
pronombre.

**Lo que NO se arregla con ninguno:** los lemas. `tenés` sigue teniendo lema
`tenés`, `sabés` lema `sabés`, `sos` lema `sos`. Ninguno de los tres modelos
llega a `tener`, `saber`, `ser`.

### Por qué esa distinción es la que importa

Si contás **categorías gramaticales** —cuántos verbos, cuántos sustantivos—
pasar de `sm` a `lg` te arregla el análisis.

Si contás **lemas** —cuántas veces aparece el verbo *tener*— no te arregla nada,
porque en tu corpus el voseo va a seguir contando aparte.

Comprobémoslo sobre el corpus real.

In [ ]:
formas_de_vos = ["tenés", "sabés", "sos", "querés", "podés", "venís", "hacés", "decís"]

nlp = modelos["lg"]
primeros = corpus.head(800)
muestra_grande = list(primeros["texto"])

componentes_apagados = ["parser", "ner"]  # disable=: no los necesitamos acá

encontradas = Counter()
for documento in nlp.pipe(muestra_grande, batch_size=100, disable=componentes_apagados):
    for token in documento:
        minuscula = token.text.lower()
        if minuscula in formas_de_vos:
            lema = token.lemma_.lower()
            # Contamos cada combinación de forma, lema y categoría por separado.
            clave = (minuscula, lema, token.pos_)
            encontradas[clave] += 1

filas = []
for clave, veces in encontradas.most_common(12):
    forma = clave[0]
    lema = clave[1]
    pos = clave[2]
    fila = {"forma": forma, "lema_que_asigna": lema, "pos": pos, "veces": veces}
    filas.append(fila)

pd.DataFrame(filas)

Cada una de esas formas cuenta como una palabra distinta de su infinitivo.
En un ranking de lemas, *tener* aparece más abajo de lo que corresponde y
*tenés* aparece como si fuera otro verbo.

Se arregla con un diccionario de correcciones.

In [ ]:
CORRECCIONES_VOSEO = {
    "tenés": "tener",
    "sabés": "saber",
    "sos": "ser",
    "querés": "querer",
    "podés": "poder",
    "venís": "venir",
    "hacés": "hacer",
    "decís": "decir",
    "andá": "andar",
    "mirá": "mirar",
    "vení": "venir",
    "dale": "dar",
}

Un diccionario alcanza como modelo de datos: la forma que el modelo no reconoce, y
el infinitivo que le corresponde.

La función de abajo lo consulta primero y solo cae en el lema de spaCy si la forma
no está en la lista.

In [ ]:
def lema_corregido(token):
    """Devuelve el lema del token, corrigiendo las formas del voseo."""
    minuscula = token.text.lower()
    if minuscula in CORRECCIONES_VOSEO:
        return CORRECCIONES_VOSEO[minuscula]
    return token.lemma_.lower()

Y ahora la comparación, columna contra columna, sobre una frase con voseo.

In [ ]:
prueba = nlp("Vos tenés razón y sabés lo que hacés.")

filas = []
for token in prueba:

    # Descartamos los signos de puntuación.
    if not token.is_punct:
        lema_de_spacy = token.lemma_.lower()
        lema_con_correccion = lema_corregido(token)
        fila = {
            "token": token.text,
            "lema_spacy": lema_de_spacy,
            "lema_corregido": lema_con_correccion,
        }
        filas.append(fila)

pd.DataFrame(filas)

> **Este diccionario es tuyo y es incompleto.** Tiene doce entradas y el voseo
> tiene muchas más. Ampliarlo es parte de tu trabajo, y cada entrada que agregues
> es una decisión que tenés que poder justificar.
>
> Fijate que metí `dale` → `dar`, que es discutible: en el uso rioplatense *dale*
> muchas veces no es el verbo *dar* sino una interjección. ¿Lo dejarías?

## 3. Auditar el NER

En el cuaderno `03` viste una lista de "personas" con cosas que no eran personas.
Ahora vamos a mirarlo en serio, comparando los dos modelos extremos.

In [ ]:
primeros = corpus.head(500)
muestra_ner = list(primeros["texto"])

conteos = {}
for etiqueta in ["sm", "lg"]:
    modelo = modelos[etiqueta]
    contador = Counter()
    for documento in modelo.pipe(muestra_ner, batch_size=50):
        for entidad in documento.ents:
            # Cada entidad se cuenta como un par (texto, tipo).
            clave = (entidad.text, entidad.label_)
            contador[clave] = contador[clave] + 1
    conteos[etiqueta] = contador

distintas_sm = len(conteos["sm"])
distintas_lg = len(conteos["lg"])
print("entidades distintas con sm:", distintas_sm)
print("entidades distintas con lg:", distintas_lg)

Los dos modelos encontraron cantidades distintas de entidades. Eso todavía no dice
cuál anda mejor: puede encontrar más porque acierta más, o porque se equivoca más.

Hay que mirar **cuáles** son.

In [ ]:
# Sacamos las 12 más frecuentes de cada modelo antes de armar la tabla.
top_sm = conteos["sm"].most_common(12)
top_lg = conteos["lg"].most_common(12)

filas = []
# posicion: el lugar en los dos rankings a la vez (0 es el primero)
for posicion in range(12):
    entidad_sm, veces_sm = top_sm[posicion]
    entidad_lg, veces_lg = top_lg[posicion]

    # Cada entidad es un par (texto, tipo): [0] es el texto y [1] el tipo.
    texto_sm = entidad_sm[0]
    tipo_sm = entidad_sm[1]
    texto_lg = entidad_lg[0]
    tipo_lg = entidad_lg[1]

    fila = {
        "sm_entidad": texto_sm,
        "sm_tipo": tipo_sm,
        "sm_veces": veces_sm,
        "lg_entidad": texto_lg,
        "lg_tipo": tipo_lg,
        "lg_veces": veces_lg,
    }
    filas.append(fila)

pd.DataFrame(filas)

### Contar los errores obvios

Hay una categoría de error que se puede detectar sin saber nada del texto:
**palabras que empiezan una oración y no son nombres propios**. Adverbios,
conectores, pronombres. Si el modelo las marca como personas o lugares, se
equivocó.

In [ ]:
SOSPECHOSAS = [
    "Ahora", "Después", "Había", "Además", "También", "Así", "Entonces",
    "Ayer", "Hoy", "Mañana", "Me", "Qué", "Cuando", "Aunque", "Pero",
    "Todo", "Nada", "Esa", "Este", "Desde", "¿",
]

for etiqueta in ["sm", "lg"]:
    distintas = 0
    total = 0
    # items(): cada clave con su cantidad. La clave es el par (texto, tipo).
    for clave, veces in conteos[etiqueta].items():
        texto = clave[0]
        if texto in SOSPECHOSAS:
            distintas = distintas + 1
            total = total + veces
    print(etiqueta + ":", distintas, "entidades falsas distintas,", total, "apariciones")

Esto **no** es una medición de calidad: es un piso. Solo cuenta los errores
que pudimos detectar con una lista escrita a mano. Los errores de tipo —marcar
una organización como lugar— no aparecen acá.

Para medir de verdad hay que anotar a mano.

## 4. Medir: precisión sobre una muestra anotada

La única forma de saber qué tan bien anda el modelo **en tu corpus** es tomar
una muestra, anotarla vos, y comparar.

Vamos a hacerlo con 40 entidades. Es poco, pero alcanza para ver el orden de
magnitud.

In [ ]:
# random: para tomar una muestra reproducible.
import random

nlp = modelos["lg"]

primeros = corpus.head(300)
textos = list(primeros["texto"])

entidades_muestra = []
for documento in nlp.pipe(textos, batch_size=50):
    for entidad in documento.ents:
        # sent: la oración donde está la entidad. [:90]: sus primeros 90 caracteres
        contexto = entidad.sent.text[:90]
        registro = {
            "texto": entidad.text,
            "tipo": entidad.label_,
            "contexto": contexto,
        }
        entidades_muestra.append(registro)

# seed(42): fija el azar, para que la mezcla salga siempre igual
random.seed(42)
random.shuffle(entidades_muestra)  # shuffle(): mezcla la lista, como un mazo de cartas
para_anotar = entidades_muestra[:40]

cantidad_para_anotar = len(para_anotar)
print("entidades a revisar:", cantidad_para_anotar)


pd.DataFrame(para_anotar[:10])

### Tu turno

Revisá las 40 y decidí, para cada una, si el modelo acertó. Corré la celda de
abajo para verlas todas, y completá la lista `MIS_ANOTACIONES` con `True` o
`False`.

**Sí, es tedioso.** Ese tedio es el punto: es lo que cuesta tener un número
confiable, y es la razón por la que casi nadie lo hace y casi todos publican
resultados sin saber cuánto valen.

In [ ]:
# set_option("display.max_rows", 40): que pandas muestre hasta 40 filas sin cortar
pd.set_option("display.max_rows", 40)
pd.DataFrame(para_anotar)

In [ ]:
# Poné True si el modelo acertó (el texto ES una entidad Y el tipo es correcto),
# False si se equivocó. Tiene que tener 40 valores.
MIS_ANOTACIONES = []

cantidad_anotada = len(MIS_ANOTACIONES)

if cantidad_anotada != 40:
    print("Faltan anotaciones:", cantidad_anotada, "de 40")
else:
    # sum() sobre True y False cuenta los True: cada True vale 1
    aciertos = sum(MIS_ANOTACIONES)
    precision = 100 * aciertos / 40
    print("aciertos :", aciertos, "de 40")
    # :.0f: sin decimales
    print("precisión:", f"{precision:.0f}%")

> ### Perilla
>
> Cambiá `random.seed(42)` por otro número y volvé a anotar 40 entidades
> distintas. ¿Te da parecido?
>
> Si con dos muestras de 40 obtenés 60% y 80%, entonces 40 no alcanza. La
> pregunta "¿cuántos casos necesito anotar?" no tiene respuesta general: se
> contesta midiendo la variación entre muestras.
>
> Esa negociación entre lo que cuesta anotar y la confianza que te da el número
> es el trabajo real de cualquiera que use estas herramientas.

## 5. ¿Precisión para quién?

Recién sacaste un número de precisión sobre una muestra. Ese número es un promedio, y
un promedio no dice **sobre qué se equivoca**.

Hasta acá el cuaderno mostró una sola forma de no-neutralidad: el modelo entiende peor el
voseo. Es cierto y está medido, pero es la mitad más inofensiva del asunto. La otra
mitad es **qué nombres reconoce y qué nombres no**.

### Cómo se prueba eso

Con una **prueba de pares mínimos**: la misma oración, exactamente igual, cambiando una
sola cosa. Si el resultado cambia, la causa es esa cosa y no otra.

*Par mínimo* es un término de la lingüística: dos formas que difieren en un solo
elemento. En fonología son *pata* y *bata*, que prueban que /p/ y /b/ son sonidos
distintos en castellano. Acá la idea es la misma, aplicada a un modelo en vez de a un
oído.

Lo único que cambia es el apellido.

In [ ]:
# Los apellidos son el único dato que varía. El origen es para poder agrupar después.
APELLIDOS = [
    {"apellido": "García", "origen": "castellano"},
    {"apellido": "Fernández", "origen": "castellano"},
    {"apellido": "Rossi", "origen": "italiano"},
    {"apellido": "Bianchi", "origen": "italiano"},
    {"apellido": "Mamani", "origen": "aimara"},
    {"apellido": "Quispe", "origen": "quechua"},
    {"apellido": "Condori", "origen": "aimara"},
    {"apellido": "Choque", "origen": "aimara"},
    {"apellido": "Nazar", "origen": "árabe"},
    {"apellido": "Yapur", "origen": "árabe"},
]

cantidad_de_apellidos = len(APELLIDOS)
print("apellidos en la prueba:", cantidad_de_apellidos)

Los diez son apellidos frecuentes en la Argentina. Los cuatro de origen quechua y aimara
lo son especialmente en el noroeste y en el conurbano bonaerense.

La función de abajo arma la oración, la procesa y devuelve con qué etiqueta el modelo vio
ese apellido. El nombre de pila se mantiene fijo: así lo único que varía entre una
oración y la siguiente es el apellido.

In [ ]:
def como_lo_ve(modelo, apellido):
    """Devuelve la etiqueta con que el modelo reconoce ese apellido, y qué texto abarcó."""
    oracion = f"Ayer María {apellido} llegó tarde a la reunión y pidió disculpas."
    documento = modelo(oracion)
    for entidad in documento.ents:
        if apellido in entidad.text:
            # Devuelve dos valores juntos: la etiqueta y el texto de la entidad.
            return entidad.label_, entidad.text
    return "—", ""


resultado = como_lo_ve(modelo=modelos["lg"], apellido="García")
print(resultado)

Ahora la prueba completa, con el modelo chico y el grande lado a lado.

In [ ]:
modelo_lg = modelos["lg"]
modelo_sm = modelos["sm"]

filas = []
for entrada in APELLIDOS:
    apellido = entrada["apellido"]
    # como_lo_ve() devuelve dos valores: los guardamos en dos variables a la vez.
    etiqueta_lg, texto_lg = como_lo_ve(modelo=modelo_lg, apellido=apellido)
    etiqueta_sm, texto_sm = como_lo_ve(modelo=modelo_sm, apellido=apellido)
    fila = {
        "apellido": apellido,
        "origen": entrada["origen"],
        "sm": etiqueta_sm,
        "lg": etiqueta_lg,
        "lg abarcó": texto_lg,
    }
    filas.append(fila)

pd.DataFrame(filas)

### Cómo se lee esa tabla

**No te digo el resultado.** Corré la celda y mirála, porque la conclusión depende de lo
que efectivamente salga.

Hay tres cosas que pueden aparecer, y cada una significa algo distinto:

| Lo que ves | Qué significa |
|---|---|
| Una raya `—` | El modelo no vio ninguna entidad ahí. Para él ese apellido no es un nombre |
| `PER` | Lo reconoció como persona. Es lo correcto |
| `LOC`, `ORG` o `MISC` | Lo vio, pero lo clasificó mal |
| La columna *lg abarcó* | Si dice solo `María`, el modelo cortó el nombre a la mitad |

Si las diez filas dan `PER`, la prueba no encontró nada y eso también es un resultado que
hay que reportar. Si aparecen diferencias, fijate **si se agrupan por origen** o si están
repartidas al azar.

### Un detalle que se coló

Mirá la columna `lg abarcó`. En las diez filas dice **«Ayer María ‹apellido›»**, con el
`Ayer` adentro de la entidad.

Ese `Ayer` no es parte de ningún nombre. Es exactamente el error que contaste en la
sección 3: una palabra que abre la oración, va en mayúscula, y el modelo la toma por
nombre propio. Está en la lista de `SOSPECHOSAS` que usaste hace un rato.

Apareció solo, en una oración que escribimos nosotros para probar otra cosa. Así de fácil
se cuela.

Como el error es **el mismo en las diez filas**, no arruina la comparación: lo único que
varía entre una fila y otra sigue siendo el apellido. Pero sirve de recordatorio: el
material de prueba también hay que auditarlo.

### Por qué pasaría

`es_core_news_lg` se entrenó con dos corpus, y los dos están documentados:

- **UD Spanish AnCora**, que es texto periodístico, mayoritariamente peninsular.
- **WikiNER**, derivado de Wikipedia.

Los vectores vienen de fastText sobre OSCAR Common Crawl y Wikipedia.

Un modelo estadístico reconoce lo que vio muchas veces. Si un apellido aparece poco en la
prensa española y poco en Wikipedia, el modelo tiene menos evidencia para tratarlo como
nombre propio. **Lo que el modelo reconoce peor es lo que estaba menos representado en el
texto con que se lo entrenó.**

### Lo que dio cuando la corrimos

Con `es_core_news_lg`, los diez apellidos dieron `PER`. Los diez.

Con `es_core_news_sm`, fallaron tres: *Bianchi* y *Yapur* como `ORG`, y *Nazar* sin
reconocer.

**La sospecha con la que armamos la prueba no se confirmó.** Estábamos mirando de reojo
los apellidos de origen quechua y aimara, y los cuatro dieron `PER` en los dos modelos.
Los que fallaron fueron los dos de origen árabe y uno italiano, y solo en el modelo chico.

Eso es un resultado y hay que reportarlo así. **Una hipótesis que no se confirma no es un
experimento fallido**: es lo que pasa la mayor parte de las veces, y es exactamente la
razón por la que se mide en vez de suponer. Si hubiéramos escrito la conclusión antes de
correr la celda, habríamos escrito algo falso.

Lo que sí queda en pie es más chico y más firme: **el modelo chico no se equivoca al
azar**. De diez apellidos falla en tres, y ninguno de los tres es castellano.

Si tu corrida da otra cosa, no está mal: los modelos cambian de versión. Compará y anotá
la diferencia, que también es un dato.

> **Esto es una prueba exploratoria, no una medición de desempeño.** Diez apellidos y una sola oración no miden el
> desempeño de nadie. Sirven para formular una sospecha y mostrar cómo se la pone a
> prueba. Un resultado negativo indica un hueco en los datos de entrenamiento, no una
> intención de nadie: nadie programó al modelo para esto.

> ### Perilla
>
> Cambiá la oración de `como_lo_ve` y volvé a correr la prueba.
>
> - ¿Qué pasa si sacás el nombre de pila y dejás el apellido solo?
> - ¿Y si en vez de *«Ayer»* la oración empieza directamente con el nombre?
> - ¿Y si agregás un cargo adelante, como *«la ingeniera»*?
>
> **Antes de correr cada variante, escribí qué esperás que cambie.** Si el resultado se
> mueve tanto con cambios así de chicos, eso también es un hallazgo sobre el modelo.

### Y esto conecta con lo que sigue

La sección que viene muestra cómo un error de etiquetado llega hasta la conclusión.
Ahora ya sabés que esos errores **no se reparten parejo**. Un error que cae siempre sobre
los mismos apellidos no es ruido: es un sesgo, y viaja hasta el final del análisis.

## 6. Cómo se propaga un error

Un error de etiquetado no se queda quieto: viaja hasta tu conclusión.

Supongamos que querés saber quiénes son los personajes más mencionados del
corpus. Hacés lo obvio: contar entidades `PER`.

In [ ]:
conteo_personas = Counter()
# items(): cada clave con su cantidad. La clave es el par (texto, tipo).
for clave, veces in conteos["lg"].items():
    texto = clave[0]
    tipo = clave[1]
    if tipo == "PER":
        conteo_personas[texto] += veces

filas = []
for nombre, veces in conteo_personas.most_common(12):
    fila = {"persona_segun_el_modelo": nombre, "veces": veces}
    filas.append(fila)

pd.DataFrame(filas)

Si publicás esa tabla, publicás los errores del modelo con tu firma.

Y fijate en algo más sutil que un falso positivo: **el mismo personaje aparece
con nombres distintos**, y cada variante se cuenta por separado.

In [ ]:
# Buscamos las variantes de un mismo personaje: los nombres que contienen "Mirta" o "Bertotti".
for nombre in conteo_personas:
    if "Mirta" in nombre or "Bertotti" in nombre:
        veces = conteo_personas[nombre]
        print(" ", nombre, "->", veces)

*Mirta*, *Bertotti* y *Mirta Bertotti* son el mismo personaje —la
protagonista de *Más respeto, que soy tu madre*— y el modelo los cuenta como
tres. Sumados serían bastante más frecuentes de lo que el ranking sugiere.

En el cuaderno `05` vamos a arreglar las dos cosas: primero con reglas
explícitas, después metiendo esas reglas adentro del pipeline.

## 7. Quién construye PLN acá

Todo el cuaderno mostró lo mismo: la herramienta no fue hecha para este castellano. Falta
la otra mitad de esa frase, que es **quién sí está haciendo herramientas para él**.

### El dato que cierra el cuaderno

spaCy distribuye un cuarto pipeline para español que no usamos acá: `es_dep_news_trf`,
basado en transformers. Es el más preciso de los cuatro.

Ese pipeline está construido sobre **BETO**, un modelo entrenado en el Departamento de
Ciencias de la Computación de la **Universidad de Chile**.

Y no trae `ner`. Sus componentes son `transformer`, `morphologizer`, `parser`,
`attribute_ruler` y `lemmatizer`.

O sea: el mejor modelo de español que distribuye spaCy se hizo en la región, y
justamente no hace la tarea que este cuaderno acaba de auditar.

### Qué hay, quién lo hizo y dónde

| Recurso | Quién | Dónde | Para qué |
|---|---|---|---|
| **BETO** `dccuchile/bert-base-spanish-wwm-cased` | Cañete, Chaperon, Fuentes, Ho, Kang y Pérez | Depto. de Ciencias de la Computación, Universidad de Chile | Modelo base de español. Es el que usa `es_dep_news_trf` |
| **RoBERTuito** `pysentimiento/robertuito-base-uncased` | Pérez, Furman, Alonso Alemany y Luque | ICC, CONICET–UBA y FAMAF, Universidad Nacional de Córdoba | Entrenado sobre más de 500 millones de tuits: lengua de uso, no prensa |
| **pysentimiento** | Pérez, Rajngewerc, Giudici, Furman, Luque, Alonso Alemany y Martínez | Mismos grupos | Librería terminada de opinión y NLP social. Español, inglés, italiano y portugués |

Y para que la comparación no quede en Norte contra Sur: `PlanTL-GOB-ES/roberta-base-bne`
lo hizo el Gobierno de **España** sobre el corpus de la Biblioteca Nacional. También es
regional. La pregunta nunca es de qué país sale un modelo, sino **con qué texto se
entrenó**.

### Las limitaciones, que también hay

- **Ninguno de estos es un modelo de spaCy.** Son de `transformers`. Meterlos en esta
  unidad rompería la promesa de una sola librería, así que acá los nombramos y no los
  instalamos.
- **RoBERTuito está entrenado sobre Twitter.** Rinde en lengua de red social. Sobre prosa
  literaria como la de este corpus no hay garantía de que ande mejor que `lg`.
- **BETO es de 2020** y su última actualización es de enero de 2024. No es un modelo en
  desarrollo activo.

Ninguna de esas limitaciones cambia el punto: existen, tienen autoría y tienen
institución. Se pueden leer, se pueden citar y se les puede escribir.

## Lo que hiciste hoy

- Comparaste los tres modelos y viste que el chico no es más rápido, solo peor.
- Aprendiste la distinción que gobierna toda la unidad: **subir de modelo arregla
  el POS y no arregla el lema.**
- Auditaste la salida del NER sobre tu propio corpus.
- Anotaste una muestra a mano y sacaste un número de precisión.
- **Armaste una prueba de pares mínimos** para preguntar de quién es esa precisión, y
  viste que un promedio esconde sobre qué se equivoca el modelo.
- Viste cómo un error de etiquetado llega hasta la conclusión, y por qué importa que esos
  errores no se repartan parejo.
- **Supiste que existe gente en la región construyendo estas herramientas**, con nombre,
  institución y limitaciones.

## Lo que viene

En el `05` dejamos de mirar los errores y empezamos a corregirlos: reglas,
componentes propios, y un pipeline configurado por vos que se puede guardar.